# Expo Memory Synth: بناء المشهد 3D من فيديو

هذا النوتبوك ياخذ فيديو مصور بالجوال، ويطلع منه ملف Gaussian Splatting (`.ply`) تقدرين تعرضينه بالموقع.

**الخطوات:** فيديو ← فريمات ← (طمس الوجيه) ← COLMAP يحسب أماكن الكاميرا ← تدريب gsplat ← ملف `.ply`

**قبل ما تبدين:**
- من القائمة: `Runtime > Change runtime type` واختاري **T4 GPU**.
- ارفعي الفيديو على Google Drive داخل مجلد اسمه `expo-memory-synth`.

## 1. تأكدي إن فيه GPU

In [ ]:
!nvidia-smi

## 2. ربط Google Drive وتحديد المسارات
عدلي `VIDEO_NAME` باسم فيديوك. كل النتايج تنحفظ بالـ Drive عشان ما تضيع لو انقطعت الجلسة.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/expo-memory-synth'
VIDEO_NAME = 'capture.mp4'          # <-- غيري هذا لاسم فيديوك
SCENE_NAME = 'scene1'               # اسم المشهد (مجلد النتايج)

VIDEO = f'{PROJECT}/{VIDEO_NAME}'
SCENE = f'/content/{SCENE_NAME}'    # الشغل يصير هنا (أسرع من الـ Drive)
OUT   = f'{PROJECT}/outputs/{SCENE_NAME}'
os.makedirs(f'{SCENE}/images', exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print('Video exists:', os.path.exists(VIDEO))

## 3. تثبيت الأدوات
- **ffmpeg** لتقطيع الفيديو، و**COLMAP** لحساب أماكن الكاميرا.
- **gsplat** للتدريب، مثبت على إصدار 1.5.3 عشان الأمثلة تطابق المكتبة.
- أول تشغيل للتدريب يسوي compile لأكواد CUDA وممكن ياخذ 5-10 دقايق، هذا طبيعي.
- **إذا كولاب طلب Restart session بعد التثبيت:** سويه، وبعدين شغلي خلية 2 من جديد وكملي من خلية 4.

In [ ]:
GSPLAT_VERSION = '1.5.3'
!apt-get -qq update && apt-get -qq install -y ffmpeg colmap > /dev/null
!pip -q install gsplat=={GSPLAT_VERSION}
!rm -rf /content/gsplat && git clone -q --depth 1 --branch v{GSPLAT_VERSION} https://github.com/nerfstudio-project/gsplat.git /content/gsplat
!pip -q install -r /content/gsplat/examples/requirements.txt
!colmap -h | head -1
print('done')

## 4. تقطيع الفيديو لفريمات
- `FPS = 3` يعني 3 صور لكل ثانية. فيديو 40 ثانية = 120 صورة تقريباً.
- إذا الفيديو من يوتيوب وتبين جزء معين، عبي `START` و `END`.

In [ ]:
FPS   = 3
WIDTH = 1600
START = ''   # مثال: '00:02:15'  (اتركيه فاضي للفيديو كامل)
END   = ''   # مثال: '00:02:45'

trim = (f'-ss {START} ' if START else '') + (f'-to {END} ' if END else '')
!rm -f {SCENE}/images/*.jpg
!ffmpeg -hide_banner -loglevel error {trim} -i "{VIDEO}" -vf "fps={FPS},scale='min({WIDTH},iw)':-2" -q:v 2 {SCENE}/images/frame_%04d.jpg

n = len(os.listdir(f'{SCENE}/images'))
print(f'{n} frames')
if n < 60:  print('قليل! زيدي FPS أو استخدمي مقطع أطول')
if n > 300: print('كثير! قللي FPS عشان COLMAP ما يطول')

## 5. (اختياري) طمس الوجيه للخصوصية

In [ ]:
import cv2, glob
det = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
total = 0
for f in sorted(glob.glob(f'{SCENE}/images/*.jpg')):
    img = cv2.imread(f)
    faces = det.detectMultiScale(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 1.1, 5, minSize=(30, 30))
    for (x, y, w, h) in faces:
        img[y:y+h, x:x+w] = cv2.GaussianBlur(img[y:y+h, x:x+w], (51, 51), 0)
    if len(faces):
        cv2.imwrite(f, img); total += len(faces)
print('faces blurred:', total)

## 6. COLMAP: حساب مكان الكاميرا لكل صورة
هذي أهم خطوة. تاخذ 10-30 دقيقة حسب عدد الصور.

نستخدم `sequential_matcher` لأن الصور جاية من فيديو (كل صورة قريبة من اللي بعدها).

In [ ]:
!rm -rf {SCENE}/database.db {SCENE}/sparse && mkdir -p {SCENE}/sparse

!colmap feature_extractor --database_path {SCENE}/database.db --image_path {SCENE}/images \
    --ImageReader.single_camera 1 --ImageReader.camera_model OPENCV --SiftExtraction.use_gpu 0

!colmap sequential_matcher --database_path {SCENE}/database.db --SiftMatching.use_gpu 0

!colmap mapper --database_path {SCENE}/database.db --image_path {SCENE}/images --output_path {SCENE}/sparse

### تحققي من النتيجة
شوفي سطر **Registered images**. إذا قريب من عدد الفريمات، ممتاز.

إذا طلع أكثر من مجلد داخل `sparse` (0، 1، ...)، يعني COLMAP قسم المشهد لأجزاء، وغالباً السبب حركة سريعة أو مكان فاضي بالتصوير.

In [ ]:
!ls {SCENE}/sparse
!colmap model_analyzer --path {SCENE}/sparse/0 2>&1 | grep -E "Cameras|Images|Registered|Points|reprojection"
!cp -r {SCENE}/sparse "{OUT}/"   # نسخة احتياطية بالـ Drive

## 7. تدريب الـ Gaussian Splatting
- `MAX_STEPS = 7000` يعطي نتيجة كويسة بحوالي 15-30 دقيقة على T4. لو تبين جودة أعلى جربي 15000.
- إذا طلع خطأ بالـ flags، شيكي README حق gsplat لأن الأسماء أحياناً تتغير بين الإصدارات.

In [ ]:
MAX_STEPS = 7000
RESULTS = f'/content/results/{SCENE_NAME}'

%cd /content/gsplat/examples
!python simple_trainer.py default \
    --data_dir {SCENE} --data_factor 1 --result_dir {RESULTS} \
    --max_steps {MAX_STEPS} --save_ply --ply_steps {MAX_STEPS} --disable_viewer
%cd /content

## 8. حفظ ملف `.ply` بالـ Drive وتنزيله

In [ ]:
import glob, shutil
plys = sorted(glob.glob(f'{RESULTS}/**/*.ply', recursive=True))
print(plys)
if plys:
    final = f'{OUT}/{SCENE_NAME}.ply'
    shutil.copy(plys[-1], final)
    print('Saved to Drive:', final, f'({os.path.getsize(final)/1e6:.1f} MB)')
    from google.colab import files
    files.download(final)
else:
    print('ما لقيت ملف ply. شيكي مخرجات خلية التدريب.')

## 9. الخطوة الجاية
1. افتحي الملف بـ **SuperSplat** (https://superspl.at/editor) وقصي الأطراف المشوهة، وصدريه من جديد.
2. حطيه بالريبو داخل `web/` باسم `scene.ply` وشغلي الموقع (التفاصيل بالـ README).
3. شغلي `scripts/narrate.py` و `scripts/make_audio.py` عشان السرد والصوت.